In [1]:
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler

In [2]:
true_b = 1
true_w = 2
N = 100

# data generation
np.random.seed(42)
x = np.random.rand(N, 1)
epsilon = (.1 * np.random.randn(N, 1))
y = true_b + true_w * x + epsilon

In [3]:
# shuffles the indices
idx = np.arange(N)
np.random.shuffle(idx)

# uses first 80 random indices for train
train_idx = idx[:int(N* .8)]
# uses the remaining indices for validation
val_idx = idx[int(N* .8):]

# generates train and validation sets
x_train, y_train = x[train_idx], y[train_idx]
x_val, y_val = x[val_idx], y[val_idx]

In [4]:
# step 0 - initialzes parameters "b" and "w" randomly
np.random.seed(42)
b = np.random.randn(1)
w = np.random.randn(1)

print(b, w)

[0.49671415] [-0.1382643]


In [5]:
# step 1 - computes our model's predicted output - forward pass
yhat = b + w * x_train

In [6]:
# step 2 - computing the loss
# we using all data point, so this is batch gradient
# descent how wrong is our model? That's the error?
error = (yhat - y_train)

# it is a regression, so it computes mean squared error (MSE)
loss = (error ** 2).mean()

print(loss)

2.7421577700550976


In [7]:
# we have to split the ranges in 100 evenly spaced intervals each
b_range = np.linspace(true_b - 3, true_b + 3, 101)
w_range = np.linspace(true_w - 3, true_w + 3, 101)
# meshgrid is a handy function that generates a grid of b and w
# values for all combinations
bs, ws = np.meshgrid(b_range, w_range)
bs.shape, ws.shape

((101, 101), (101, 101))

In [8]:
bs

array([[-2.  , -1.94, -1.88, ...,  3.88,  3.94,  4.  ],
       [-2.  , -1.94, -1.88, ...,  3.88,  3.94,  4.  ],
       [-2.  , -1.94, -1.88, ...,  3.88,  3.94,  4.  ],
       ...,
       [-2.  , -1.94, -1.88, ...,  3.88,  3.94,  4.  ],
       [-2.  , -1.94, -1.88, ...,  3.88,  3.94,  4.  ],
       [-2.  , -1.94, -1.88, ...,  3.88,  3.94,  4.  ]], shape=(101, 101))

In [9]:
dummy_x = x_train[0]
dummy_yhat = bs + ws * dummy_x
dummy_yhat.shape

(101, 101)

In [10]:
all_predictions = np.apply_along_axis(
    func1d=lambda x: bs + ws * x,
    axis=1,
    arr=x_train,
)
all_predictions.shape

(80, 101, 101)

In [11]:
all_labels = y_train.reshape(-1, 1, 1)
all_labels.shape

(80, 1, 1)

In [12]:
all_errors = (all_predictions - all_labels)
all_errors.shape

(80, 101, 101)

In [13]:
all_losses = (all_errors ** 2).mean(axis=0)
all_losses.shape

(101, 101)

In [14]:
# step 3 - computes gradients for both "b" and "w" parameters
b_grad = 2 * error.mean()
w_grad = 2 * (x_train * error).mean()
print(b_grad, w_grad)

-3.044811379650508 -1.8337537171510832


In [15]:
# sets learning rate - this is "eta" - the "n" - like greek letter
lr = 0.1
print(b, w)

# step 4 - updates parameters using gradients and the learning rate
b = b - lr * b_grad
w = w - lr * w_grad

print(b, w)

[0.49671415] [-0.1382643]
[0.80119529] [0.04511107]


In [16]:
# bad features
true_b = 1
true_w = 2
N = 100

# data generation
np.random.seed(42)

# divide w by 10
bad_w = true_w / 10
# multiply x by 10
bad_x = np.random.rand(N, 1) * 10

# so, the net effect on y is zero - it is still
y = true_b + bad_w * bad_x + (.1 * np.random.randn(N, 1))

In [17]:
bad_x_train, y_train = bad_x[train_idx], y[train_idx]
bad_x_val, y_val = bad_x[val_idx], y[val_idx]

In [18]:
scaler = StandardScaler(with_mean=True, with_std=True)
# the train set only to fit the scaler
scaler.fit(x_train)

# fit scaler to transform
# both train and validation sets
scaled_x_train = scaler.transform(x_train)
scaled_x_val = scaler.transform(x_val)